In [1]:
import spacy
nlp = spacy.load("en_core_web_lg")

In [2]:
sentence = "The revenue in 2022 was 100 million dollars."
doc = nlp(sentence)

print("Entities detected by spaCy:")
for ent in doc.ents:
    print(f"Text: {ent.text}, Label: {ent.label_}")

Entities detected by spaCy:
Text: 2022, Label: DATE
Text: 100 million dollars, Label: MONEY


In [3]:
def extract_monetary_values(sentence):
    """
    Extracts monetary values from text using spaCy.
    """
    doc = nlp(sentence)
    return [ent.text for ent in doc.ents if ent.label_ == "MONEY"]

# Test it
sentence = "The revenue in 2022 was 100 million dollars."
print("Extracted Money Values:", extract_monetary_values(sentence))


Extracted Money Values: ['100 million dollars']


In [4]:
def extract_years(sentence):
    """
    Extracts years using spaCy NER and fixes cases where years are labeled as CARDINAL.
    """
    doc = nlp(sentence)
    years = []

    for ent in doc.ents:
        if ent.label_ == "DATE" and ent.text.isdigit() and len(ent.text) == 4:
            years.append(ent.text)
    return years

# Test it
sentence = "Our profit for the year was 24.0 million dollars, 372.0 million dollars and 1,096.4 million dollars in 2021, 2022 and 2023, respectively."
print("Extracted Years:", extract_years(sentence))


Extracted Years: ['2021', '2022', '2023']


In [5]:
def extract_companies(sentence):
    """
    Extracts company names using spaCy's NER.
    """
    doc = nlp(sentence)
    return [ent.text for ent in doc.ents if ent.label_ == "ORG"]

sentence = "In 2023, Apple’s revenue grew to 200 billion dollars, while Microsoft reported 180 billion dollars."
print("Extracted Companies:", extract_companies(sentence))


Extracted Companies: ['Apple', 'Microsoft']


In [6]:
def extract_financial_properties(sentence):
    """
    Extracts financial properties using a manual list of financial terms while filtering out company names.
    
    :param sentence: The input financial text.
    :return: List of detected financial properties.
    """
    # 🔹 Define a manual list of common financial terms
    FINANCIAL_TERMS = {
        "revenue", "net profit", "gross profit", "operating expenses", 
        "total expenditure", "EBITDA", "earnings", "income", "costs", "profit margin"
    }

    doc = nlp(sentence)
    financial_terms = []
    company_names = {ent.text for ent in doc.ents if ent.label_ == "ORG"}  # Extract company names

    for chunk in doc.noun_chunks:
        # Remove possessives ('s)
        text = chunk.text.replace("’s", "").replace("'s", "")

        words = text.split()  # Split chunk into words
        if words[0] in company_names:  # Remove company names
            text = " ".join(words[1:])

        for term in FINANCIAL_TERMS:
            if term in text.lower():  # Check if full term appears in text
                financial_terms.append(term)
                break  # Avoid duplicate matches

    return financial_terms

# Test it
sentence = "Google’s net profit in 2022 was 50 billion dollars"
print("Extracted Financial Properties:", extract_financial_properties(sentence))


Extracted Financial Properties: ['net profit']


In [7]:
def extract_change_indicators(sentence):
    """
    Extracts words indicating financial changes such as increase, decrease, and growth.
    Uses POS tagging and dependency parsing to generalize detection.
    
    :param sentence: The input financial text.
    :return: List of change indicator words.
    """
    doc = nlp(sentence)
    change_words = []

    for token in doc:
        if token.pos_ in ["VERB", "ADJ"] and token.dep_ in ["ROOT", "amod", "acomp"]:
            change_words.append(token.text)

    return change_words

# Test it
sentence = "In 2023, Apple’s revenue grew significantly to $200B, while Microsoft reported a 5% decline."
print("Extracted Change Indicators:", extract_change_indicators(sentence))

Extracted Change Indicators: ['grew']


In [8]:
def extract_relationships(sentence):
    """
    Extracts structured financial relationships from a sentence.
    If the first relationship has a year but following ones do not, they inherit the same year.
    
    Returns: List of (company, financial_property, year, monetary_value)
    """
    companies = extract_companies(sentence)
    years = extract_years(sentence)
    financial_properties = extract_financial_properties(sentence)
    monetary_values = extract_monetary_values(sentence)
    # print("1", financial_properties)

    relationships = []
    used_values = set(
    last_year = None  
    last_financial_property = None 

    for i in range(max(len(companies), len(financial_properties), len(monetary_values))):
        company = companies[i] if i < len(companies) else None
        year = years[i] if i < len(years) else last_year  # Inherit year if missing
        financial_property = financial_properties[i] if i < len(financial_properties) else last_financial_property
        monetary_value = monetary_values[i] if i < len(monetary_values) else None
        # print(company, year, financial_property, monetary_value)

        # Ensure monetary values are not duplicated
        if monetary_value in used_values:
            monetary_value = None
        else:
            used_values.add(monetary_value)

        # Update last known year and financial property
        if year:
            last_year = year
        if financial_property:
            last_financial_property = financial_property

        # Ensure we store valid relationships
        if company and financial_property and monetary_value:
            relationships.append((company, financial_property, year, monetary_value))
    print(relationships)

    return relationships

In [9]:
test_sentences = [
    "In 2023, Apple’s revenue grew to 200 billion dollars, while Microsoft reported 180 billion dollars.",
    "Google’s net profit in 2022 was 50 billion dollars.",
    "Amazon’s operating expenses in 2021 totaled $150B."
]

for sentence in test_sentences:
    print(f"\nSentence: {sentence}")
    print(f"Extracted Relationships: {extract_relationships(sentence)}")


Sentence: In 2023, Apple’s revenue grew to 200 billion dollars, while Microsoft reported 180 billion dollars.
[('Apple', 'revenue', '2023', '200 billion dollars'), ('Microsoft', 'revenue', '2023', '180 billion dollars')]
Extracted Relationships: [('Apple', 'revenue', '2023', '200 billion dollars'), ('Microsoft', 'revenue', '2023', '180 billion dollars')]

Sentence: Google’s net profit in 2022 was 50 billion dollars.
[('Google', 'net profit', '2022', '50 billion dollars')]
Extracted Relationships: [('Google', 'net profit', '2022', '50 billion dollars')]

Sentence: Amazon’s operating expenses in 2021 totaled $150B.
[('Amazon', 'operating expenses', '2021', '150B.')]
Extracted Relationships: [('Amazon', 'operating expenses', '2021', '150B.')]
